# 1. Weight decay

Loss function hiện tại:
$$\mathcal{L}(\mathbf{W}) = \frac{1}{n}\sum_{i=1}^{n}\ell(f(\mathbf{x}_i, \mathbf{W}), y_i)$$

Thêm weight decay:
$$\mathcal{L}_{\text{WD}}(\mathbf{W}) = \underbrace{\frac{1}{n}\sum_{i=1}^{n}\ell(f(\mathbf{x}_i, \mathbf{W}), y_i)}_{\text{Loss gốc (data fit)}} + \underbrace{\frac{\lambda}{2}\|\mathbf{W}\|^2}_{\text{Penalty (phạt W lớn)}}$$


| Ký hiệu            | Ý nghĩa                                            | Ví dụ                                                       |
| ------------------ | -------------------------------------------------- | ----------------------------------------------------------- |
| $\mathcal{L}$      | Loss function                                      | Cross-entropy                                               |
| $\|\mathbf{W}\|^2$ | Tổng bình phương **tất cả** trọng số: $\sum w_i^2$ | Nếu W = [3, -2, 1] thì $\|W\|^2 = 9 + 4 + 1 = 14$           |
| $\lambda$ (lambda) | **Hệ số phạt** — càng lớn → kìm W càng mạnh        | Thường $10^{-4}$ đến $10^{-2}$                              |
| $\frac{1}{2}$      | Hệ số cho công thức đạo hàm gọn hơn                | Đạo hàm $\frac{\lambda}{2}w^2$ = $\lambda w$ (không cần ×2) |

In [2]:
import torch
from torch import nn
from torch.nn import functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

# ====== DATA ======
trans = transforms.Compose([transforms.ToTensor()])
train_data = torchvision.datasets.FashionMNIST(
    './.data', train=True, transform=trans, download=True)
test_data = torchvision.datasets.FashionMNIST(
    './.data', train=False, transform=trans, download=True)
train_loader = DataLoader(train_data, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=256, shuffle=False)

# ====== PARAMETERS ======
num_inputs, num_hiddens, num_outputs = 784, 256, 10
W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens))
W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs))
params = [W1, b1, W2, b2]


In [3]:
import torch
from torch import nn


model = nn.Sequential(
    nn.Flatten(),
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)

In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-3)


In [6]:
for epoch in range(10):
    model.train()
    for X, y in train_loader:
        loss = F.cross_entropy(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Evaluate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in test_loader:
            correct += (model(X).argmax(1) == y).sum().item()
            total += y.shape[0]
    print(f"Epoch {epoch+1:2d} | Test Acc: {correct/total:.4f}")

Epoch  1 | Test Acc: 0.8241
Epoch  2 | Test Acc: 0.8065
Epoch  3 | Test Acc: 0.8198
Epoch  4 | Test Acc: 0.8241
Epoch  5 | Test Acc: 0.8415
Epoch  6 | Test Acc: 0.8423
Epoch  7 | Test Acc: 0.8354
Epoch  8 | Test Acc: 0.8259
Epoch  9 | Test Acc: 0.8465
Epoch 10 | Test Acc: 0.8388


In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.LazyLinear(256), nn.ReLU(), nn.Dropout(0.5),   # ← Dropout sau ReLU!
    nn.LazyLinear(256), nn.ReLU(), nn.Dropout(0.5),   # ← Layer 2
    nn.LazyLinear(10)
)